# Main for Data Analysis 

Filters analyses and plots track and tour data

This notebook is to be executed after data/data_handling/data-aquisition.ipynb and before main_energy_sim.ipynb

### Imports 

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import data.data_handling.data_visualization as dv
import data.data_handling.data_processing as dp
from data.data_handling.data_processing import colors_fleets, alpha_major, alpha_minor
import data.data_handling.sequential_analysis as sq

from utils.config import textwidth, h_169, h_43

In [ ]:
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'sans-serif',
    'font.sans-serif': 'Arial',
    'font.size': 12,
    'text.latex.preamble': r'\usepackage{siunitx}'
})

## Preprocessing 

### Data filtering
Load and preprocess the raw trips data.

Preprocessing includes, filtering out tracks shorter than 1 km, with avg. speeds > max. speed and with negative durations. Additioally adds forwarder data to each trip.

In [ ]:
# Define LOCATIONS
LOCATIONS = dp.LOCATIONS

# Load data
df_trips_unfiltered, df_fleet= dp.load_data()
# Preprocess data
df_trips = dp.preprocess_trips_data(df_trips_unfiltered, df_fleet)

# Fixes tours that 'spill over' into the next tour and cause tracks to be assigned to the wrong track
# This is a problem in the raw data (df_trips_unfiltered)
df_trips = dp.fix_faulty_tour_assignment(df_trips, df_trips_unfiltered)

# Fixes assigns a locations that corresponds to a home base to the end of each tour, if it does not
# do so anyway. Usually because the last track of a tour has been filtered out in preprocess_trips_data()
# If new home bases are addded, update the homebase_map in fix_faulty_tour_endings()
df_trips = dp.fix_faulty_tour_endings(df_trips, df_trips_unfiltered)
#df_trips.to_csv('./input/stations/tracks_filtered.csv')

### Process truck occupations and resample dataframes

In [ ]:
df_stops = dp.process_stops_data(df_trips)

# Cumulative time spent driving or in each type of location for each truck
df_rt_joined_plot = dp.calculate_rest_times_and_driving(df_stops, df_trips)

# Resample occupation data
# shows time spent (in min) not occurencess
truck_day_occ = dp.resample_occupation_data(df_stops, df_trips)

### Create tour statistics 

Aggregate all tracks that belong to a single tour into a row 

Creates: input/stations/tours.csv

In [ ]:
df_tours = dp.aggregate_tours(df_trips, save=True)

# Calculate the percentage of tours with distance_km < 300, < 375, < 500, < 1000
distances = [300, 375, 500, 1000]
percentages = {dist: (df_tours[df_tours['distance_km'] < dist].shape[0] / df_tours.shape[0]) * 100 for dist in distances}

for dist, perc in percentages.items():
    print(f"Tours under {dist} km: {perc:.2f}%")

print('\n')  
# Calculate the percentage of tours with distance_km < 300, < 375, < 500, < 1000 per freight forwarder
ff_percentages = {}
for ff in df_tours['freight_forwarder'].unique():
    ff_data = df_tours[df_tours['freight_forwarder'] == ff]
    ff_percentages[ff] = {dist: (ff_data[ff_data['distance_km'] < dist].shape[0] / ff_data.shape[0]) * 100 for dist in distances}

# Print the results
for ff, stats in ff_percentages.items():
    print(f"Freight Forwarder {ff}:")
    for dist, perc in stats.items():
        print(f"  Tours under {dist} km: {perc:.2f}%")

### Meta data 

In [ ]:
# Calculation of general meta data
meta_data = dp.calculate_meta_data(df_trips_unfiltered, df_trips, df_fleet)
# Converting meta data to DataFrame
general_df, _, _, _ = dp.meta_data_to_df(meta_data)

# Display of DataFrame
display(general_df)

# Converting meta data to DataFrame
_, temporal_df, _, _ = dp.meta_data_to_df(meta_data)

# Display of DataFrame
display(temporal_df)

# Converting meta data to DataFrame
_, _, _, ff_df = dp.meta_data_to_df(meta_data)

# Display of DataFrame
display(ff_df)

## Plots

### Data Recordings - Weekly Distance    

In [ ]:
def plot_weekly_distances(df_trips): #TODO: rename to plot_weekly_distances and delete old version in code
    """
    Plot the weekly distances covered by the fleet as a stacked bar plot, centered on week start dates (Sunday).
    """
    
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(textwidth, h_169))

    # Preprocess
    df_trips['start_time_7d'] = pd.to_datetime(df_trips['start_time'], utc=True).dt.to_period("W-SUN")
    distances_km = df_trips.groupby(['start_time_7d', 'freight_forwarder'])['distance'].sum().unstack(fill_value=0) / 1000

    # Get first and last recording date for each fleet
    fleet_periods = df_trips.groupby('freight_forwarder').agg(
        first_date=('start_time', 'min'),
        last_date=('start_time', 'max')
    )

    # Create a color mapping for freight forwarders
    freight_forwarders = sorted(df_trips['freight_forwarder'].unique())
    colors_plot = ['#1f77b4', '#2c2c2c', '#ff7f0e', '#a0c114', '#7F7F7F', '#B3679B']
    ff_colors = {}
    for i, ff in enumerate(freight_forwarders):
        ff_colors[ff] = colors_plot[i % len(colors_plot)]

    # Plot
    distances_km.plot.bar(stacked=True, xlabel='', ylabel='distance / km', ax=ax, color=[ff_colors.get(i, '#808080') for i in distances_km.columns])

    # Configure the plot
    ax.set_ylim(0, 82000)  # Capped at 85,000 for y-axis
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10000))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(5000))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda t, pos: f"{int(t):,}"))
    ax.set_ylabel('distance / km', fontsize=14)
    ax.set_xlabel('week', fontsize=14)

    # X-axis ticks (show all weeks for readability with larger font)
    x_week_ticks = np.arange(len(distances_km))
    ax.xaxis.set_minor_locator(ticker.FixedLocator(x_week_ticks))
    ax.xaxis.set_minor_formatter(ticker.FuncFormatter(lambda t, pos: f"{distances_km.index[t].start_time.day:02d}-{distances_km.index[t].start_time.month:02d}-{distances_km.index[t].start_time.year}"))
    
    # Remove major year ticks to avoid clutter, since we add dividers
    ax.xaxis.set_major_locator(ticker.NullLocator())

    ax.tick_params(axis="y", labelsize=13)
    ax.tick_params(axis="x", which="minor", pad=20, labelrotation=90, labelsize=11)  
    ax.grid(axis="y", which='major', alpha=0.3)
    ax.grid(axis="y", which='minor', alpha=0.15)

    # Add horizontal lines for fleet recording periods
    indicator_level = 70000
    max_fleet = max(fleet_periods.index) if not fleet_periods.empty else 6  
    for i, (fleet, (first_date, last_date)) in enumerate(fleet_periods.iterrows()):
        first_week = pd.Timestamp(first_date).to_period("W-SUN").start_time
        last_week = pd.Timestamp(last_date).to_period("W-SUN").start_time
        first_idx = np.searchsorted(distances_km.index.to_timestamp(), first_week)
        last_idx = np.searchsorted(distances_km.index.to_timestamp(), last_week)
        ax.plot([first_idx, last_idx], [indicator_level + i * 2000, indicator_level + i * 2000], 
                '--|', linewidth=1, markersize=7, color=ff_colors.get(fleet, 'gray'))

    # Create year divider for 2021-2022
    year_change_idx_2021_2022 = np.searchsorted(distances_km.index.to_timestamp(), pd.Timestamp('2022-01-01').to_period("W-SUN").start_time)
    ax.axvline(x=year_change_idx_2021_2022 - 0.5, color='black', linestyle='--', alpha=0.5)

    # Create year divider for 2022-2023
    year_change_idx_2022_2023 = np.searchsorted(distances_km.index.to_timestamp(), pd.Timestamp('2023-01-01').to_period("W-SUN").start_time)
    ax.axvline(x=year_change_idx_2022_2023 - 0.5, color='black', linestyle='--', alpha=0.5)

    # Add legend (moved to upper right, entries listed vertically, kept within plot)
    ax.legend(ncol=1, title="fleet", loc='upper right', bbox_to_anchor=(1.0,0.9),fontsize=13, title_fontsize=14)

    # Adjust layout
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.25, right=0.80)  

    # Save the plot
    plt.savefig('data/output/figures/operational/data_recordings.pdf', bbox_inches='tight')
    plt.show()


plot_weekly_distances(df_trips)  

### KDE Plots - Tracks - Distance | Duration | Max Speed | Avg Speed

In [ ]:
dv.plot_kde_plots(df_trips)

### KDE Plots - Tours - Distance | Duration | Max Speed | Avg Speed

In [ ]:
dv.plot_tour_kde_plots(df_trips)

### Weekly Distance Boxplots    

In [ ]:
dv.plot_weekly_distance_boxplot(df_trips)

dv.verify_weekday_aggregation(df_trips)

### Tour Duration and Distance  


In [ ]:
dv.plot_tour_duration_distance_histogram(df_trips, max_distance=900, max_duration=35)
dv.plot_tour_duration_distance_ecdf(df_trips, max_distance=1300, max_duration=35)

In [ ]:
dv.plot_rest_time_kde(df_stops)

## Data Quality
Gaps in recording in relation to recorded distance

In [ ]:
total_signal_loss = df_trips['n_signal_loss'].sum()
avg_signal_loss_ratio = df_trips['r_signal_loss'].mean()

print(f'Total signal loss: {total_signal_loss}')
print(f'Average signal loss ratio: {avg_signal_loss_ratio}')

## Fleet occupation 

In [ ]:
dv.plot_fleet_occupation(df_rt_joined_plot, truck_day_occ)

### Average departure and arrival times

In [ ]:
dp.avg_departure_and_arrival()